In [3]:
from pyspark.sql.functions import when, col

In [4]:
%run "04Common.ipynb"

In [5]:
print(f"`{catalog}`.`{bronze}`.`raw_roads`")

`sandbox-rey-01`.`bronze`.`raw_roads`


In [6]:
dfBronzeRoad = (
    spark.readStream
        .table(f"`{catalog}`.`{bronze}`.`raw_roads`")
)

## Creating road_category_name column

In [7]:
def roadCategory(df):
    print('Creating Road Catgory Name Column', end='')
    
    dfRoadCategory = df.withColumn("Road_Category_Name",
                when(col('Road_Category') == 'TA', 'Class A Trunk Road')
                .when(col('Road_Category') == 'TM', 'Class A Trunk Motor')
                .when(col('Road_Category') == 'PA', 'Class A Principal road')
                .when(col('Road_Category') == 'PM', 'Class A Principal Motorway')
                .when(col('Road_Category') == 'M', 'Class B road')
                .otherwise('NA')
    )

    print("Success!")
    return dfRoadCategory

## Creating Road_Type

In [8]:
def roadType(df):
    print('Creating Road Catgory Name Column', end='')
    
    dfRoadType = df.withColumn("Road_Type",
                when(col('Road_Category_Name').contains('Class A'), 'Major')
                .when(col('Road_Category_Name').contains('Class B'), 'Minor')
                .otherwise('NA')
    )

    print("Success!")
    return dfRoadType

In [9]:
road_columns = dfBronzeRoad.schema.names

In [10]:
dfStaticRoad = dfBronzeRoad

dfStaticRoad = dedupDF(dfStaticRoad)
dfStaticRoad = handleNulls(dfStaticRoad, road_columns)
dfStaticRoad = roadCategory(dfStaticRoad)
dfStaticRoad = roadType(dfStaticRoad)

dfStaticRoad.schema.names

Deduplicating Dataframe
Handling NULL values for string solumnsSuccessfully handled string
Replacing Null values on Numberic ColumnsSuccessfully handled numeric value
Creating Road Catgory Name ColumnSuccess!
Creating Road Catgory Name ColumnSuccess!


['Road_ID',
 'Road_Category_Id',
 'Road_Category',
 'Region_ID',
 'Region_Name',
 'Total_Link_Length_Km',
 'Total_Link_Length_Miles',
 'All_Motor_Vehicles',
 'Road_Category_Name',
 'Road_Type']

In [12]:
dfPreview = spark.read.table(f"`{catalog}`.`{bronze}`.`raw_roads`")
dfPreview = dedupDF(dfPreview)
dfPreview = handleNulls(dfPreview, dfPreview.schema.names)
dfPreview = roadCategory(dfPreview)
dfPreview = roadType(dfPreview)

dfPreview.show(20, truncate=False)

Deduplicating Dataframe
Handling NULL values for string solumnsSuccessfully handled string
Replacing Null values on Numberic ColumnsSuccessfully handled numeric value
Creating Road Catgory Name ColumnSuccess!
Creating Road Catgory Name ColumnSuccess!
+-------+----------------+-------------+---------+-------------+--------------------+-----------------------+------------------+--------------------------+---------+
|Road_ID|Road_Category_Id|Road_Category|Region_ID|Region_Name  |Total_Link_Length_Km|Total_Link_Length_Miles|All_Motor_Vehicles|Road_Category_Name        |Road_Type|
+-------+----------------+-------------+---------+-------------+--------------------+-----------------------+------------------+--------------------------+---------+
|1      |1               |TM           |1        |South West   |301.339             |187.24                 |3.465840186E9     |Class A Trunk Motor       |Major    |
|2      |3               |TA           |1        |South West   |993.586             |

## Writing to Silver Roads

In [13]:
def writeRoadsSilverTable(StreamingDF,catalog):
    print('Writing the silver_roads Data : ',end='') 

    write_StreamSilver = (StreamingDF.writeStream
                .format('delta')
                .option('checkpointLocation',checkpoint+ "/SilverRoadsLoad/Checkpt/")
                .outputMode('append')
                .queryName("SilverRoadsWriteStream")
                .trigger(availableNow=True)
                .toTable(f"`{catalog}`.`{silver}`.`silver_roads`"))
    
    write_StreamSilver.awaitTermination()
    print(f'Writing `{catalog}`.`{silver}`.`silver_roads` Success!')

In [14]:
# Write to Silver Traffic table

writeRoadsSilverTable(dfStaticRoad, catalog)

Writing the silver_roads Data : Writing `sandbox-rey-01`.`silver`.`silver_roads` Success!
